In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
from cleantext import clean

# CONFIG

In [ ]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 5*2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paraphrasing Configuration
USE_PARAPHRASING = True
USE_TEXT_CLEANING = True
NUM_PARAPHRASES = 1
PARAPHRASE_MODEL = "t5-small"
SIMILARITY_THRESHOLD = 0.7

In [3]:
# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [6]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [7]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
def create_paraphrasing_pipeline():
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    model_name = PARAPHRASE_MODEL
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)
    model.to(DEVICE)
    return model, tokenizer

def clean_text_for_paraphrasing(text):
    """Clean text using cleantext module for better paraphrasing"""
    if pd.isna(text) or text == '':
        return text, []
    
    # Store original URLs before cleaning
    import re
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    urls = re.findall(url_pattern, text)
    
    # Replace URLs with placeholders before cleaning
    clean_text = text
    for i, url in enumerate(urls):
        clean_text = clean_text.replace(url, f" URLTOKEN{i} ")
    
    # Use cleantext to clean the text
    clean_text = clean(clean_text,
                      fix_unicode=True,
                      to_ascii=False,
                      lower=False,
                      normalize_whitespace=True,
                      no_line_breaks=True,
                      strip_lines=True,
                      keep_two_line_breaks=False,
                      no_urls=False,  # We handle URLs separately
                      no_emails=False,
                      no_phone_numbers=False,
                      no_numbers=False,
                      no_digits=False,
                      no_currency_symbols=False,
                      no_punct=False,
                      lang="en")
    
    return clean_text, urls

def restore_text_after_paraphrasing(text, urls):
    """Restore URLs after paraphrasing"""
    if pd.isna(text) or text == '':
        return text
    
    # Handle case where paraphrasing returns unexpected format
    if isinstance(text, list):
        text = text[0] if text else ""
    
    restored_text = str(text)  # Ensure it's a string
    for i, url in enumerate(urls):
        restored_text = restored_text.replace(f"URLTOKEN{i}", url)
    
    return restored_text

def paraphrase_text(text, model, tokenizer, num_return_sequences=1):
    original_text = text
    urls = []
    
    # Apply text cleaning if enabled
    if USE_TEXT_CLEANING:
        text, urls = clean_text_for_paraphrasing(text)
    
    input_text = f"paraphrase: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True, padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=512,
            num_return_sequences=num_return_sequences,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            no_repeat_ngram_size=2
        )
    
    paraphrases = []
    for output in outputs:
        paraphrase = tokenizer.decode(output, skip_special_tokens=True)
        
        # Restore URLs if cleaning was applied
        if USE_TEXT_CLEANING and urls:
            paraphrase = restore_text_after_paraphrasing(paraphrase, urls)
        
        paraphrases.append(paraphrase)
    
    return paraphrases

def get_text_similarity(text1, text2, tokenizer_sim):
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np
    
    # Simple token-based similarity for filtering
    tokens1 = set(tokenizer_sim.tokenize(text1.lower()))
    tokens2 = set(tokenizer_sim.tokenize(text2.lower()))
    
    if not tokens1 or not tokens2:
        return 0.0
    
    intersection = tokens1.intersection(tokens2)
    union = tokens1.union(tokens2)
    return len(intersection) / len(union)

def add_paraphrased_data(train_df, paraphrase_model, paraphrase_tokenizer):
    """Generate paraphrased data from training DataFrame (after train/val split)"""
    paraphrased_rows = []
    
    print(f"Starting paraphrasing augmentation on {len(train_df)} training samples...")
    print(f"Text cleaning enabled: {USE_TEXT_CLEANING}")
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Paraphrasing bodies"):
        rule = row['rule']
        body = row['body']
        label = row['label']
        rule_id = row['rule_id']
        
        # Generate paraphrases for the body only (with cleaning if enabled)
        try:
            # Only paraphrase if body has sufficient length
            if len(body.split()) > 5:
                paraphrases = paraphrase_text(body, paraphrase_model, paraphrase_tokenizer, NUM_PARAPHRASES)
                
                for paraphrase in paraphrases:
                    # Filter based on similarity threshold
                    similarity = get_text_similarity(body, paraphrase, paraphrase_tokenizer)
                    
                    if similarity > SIMILARITY_THRESHOLD and paraphrase.strip() != body.strip():
                        # Create new row with paraphrased body
                        paraphrased_text = f"{rule} [SEP] {paraphrase}"
                        paraphrased_rows.append({
                            'text': paraphrased_text,
                            'label': label,
                            'rule': rule,
                            'body': paraphrase,
                            'rule_id': rule_id
                        })
        except Exception as e:
            print(f"Error paraphrasing: {e}")
            continue
    
    # Create DataFrame from paraphrased rows
    if paraphrased_rows:
        paraphrased_df = pd.DataFrame(paraphrased_rows)
        print(f"Generated {len(paraphrased_df)} paraphrased examples")
        return paraphrased_df
    else:
        print("No paraphrased examples generated")
        return pd.DataFrame(columns=['text', 'label', 'rule', 'body', 'rule_id'])

In [ ]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data (NO paraphrasing here to avoid data leakage)
augmented_texts = (df.text.tolist() + 
                  augmented_train[0] + 
                  augmented_test[0])

augmented_labels = (df.label.tolist() + 
                   augmented_train[1] + 
                   augmented_test[1])

# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before deduplication: {augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print(f'After deduplication: {augmented_df.shape}')
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

print(f"Base dataset size (before paraphrasing): {augmented_df.shape}")
print("Paraphrasing will be applied per fold to avoid data leakage")

augmented_df.head()

In [9]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [ ]:
# Test paraphrasing functionality with proper data structure (optional)
if USE_PARAPHRASING and not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("Testing paraphrasing functionality with proper data structure...")
    
    # Create a small test sample from augmented_df
    test_sample_df = augmented_df.head(3).copy()
    print("Test sample from augmented_df:")
    print(test_sample_df[['rule', 'body', 'label']].head())
    
    # Test cleaning function
    if USE_TEXT_CLEANING:
        test_body = "This is spam content with referral links: http://spam.com/ref=123 and weird\n\nformatting!"
        cleaned_text, urls = clean_text_for_paraphrasing(test_body)
        print(f"\nCleaning test:")
        print(f"Original: {test_body}")
        print(f"Cleaned: {cleaned_text}")
        print(f"URLs found: {urls}")
    
    # Quick test with a simple model
    try:
        from transformers import T5ForConditionalGeneration, T5Tokenizer
        print(f"\nTesting paraphrasing with {PARAPHRASE_MODEL}...")
        test_model = T5ForConditionalGeneration.from_pretrained(PARAPHRASE_MODEL)
        test_tokenizer = T5Tokenizer.from_pretrained(PARAPHRASE_MODEL)
        
        # Test the new DataFrame-based function
        paraphrased_test_df = add_paraphrased_data(test_sample_df, test_model, test_tokenizer)
        
        if len(paraphrased_test_df) > 0:
            print(f"\nGenerated {len(paraphrased_test_df)} paraphrased samples:")
            print(paraphrased_test_df[['rule', 'body', 'label']].head())
        else:
            print("No paraphrased samples generated (likely due to similarity filtering)")
            
        del test_model, test_tokenizer
        torch.cuda.empty_cache()
        print("✅ Paraphrasing test completed successfully")
    except Exception as e:
        print(f"Paraphrasing test failed: {e}")
        print("Will proceed without testing - model should work on Kaggle")

In [10]:
# print(df.head())
# print(df.columns.tolist())

In [11]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [12]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [13]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [14]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    # print(rule_aucs,'Rule_AUC')
    return avg_auc_per_rule, val_loss, preds

In [15]:
from transformers import get_linear_schedule_with_warmup

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    
    # Initialize paraphrasing model once if enabled
    paraphrase_model, paraphrase_tokenizer = None, None
    if USE_PARAPHRASING:
        print("Initializing paraphrasing model for all folds...")
        paraphrase_model, paraphrase_tokenizer = create_paraphrasing_pipeline()
    
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print('--------- ','FOLD: ',fold,' --------')
        all_preds = []
        
        # Get train and validation DataFrames
        train_fold_df = augmented_df.iloc[tr_idx].copy()
        val_fold_df = augmented_df.iloc[val_idx].copy()
        
        # Apply paraphrasing augmentation to training data only (NO DATA LEAKAGE)
        if USE_PARAPHRASING and paraphrase_model is not None:
            print(f"Applying paraphrasing to fold {fold} training data...")
            paraphrased_df = add_paraphrased_data(train_fold_df, paraphrase_model, paraphrase_tokenizer)
            
            # Combine original training data with paraphrased data
            if len(paraphrased_df) > 0:
                combined_train_df = pd.concat([train_fold_df, paraphrased_df], ignore_index=True)
                print(f"Training data: {len(train_fold_df)} original + {len(paraphrased_df)} paraphrased = {len(combined_train_df)} total")
            else:
                combined_train_df = train_fold_df
                print(f"No paraphrased data generated, using {len(combined_train_df)} original samples")
        else:
            combined_train_df = train_fold_df
            print(f"Paraphrasing disabled, using {len(combined_train_df)} original training samples")
        
        # Create datasets
        val_ds = JigsawDataset(
            val_fold_df['text'].tolist(), 
            val_fold_df['label'].tolist(), 
            val_fold_df['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
    
        train_ds = JigsawDataset(
            combined_train_df['text'].tolist(), 
            combined_train_df['label'].tolist(), 
            combined_train_df['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        # Initialize classification model and load MLM pre-trained weights
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        for name, param in model.named_parameters():
            if name.startswith('base.embedding'):
                param.requires_grad = False
                
        print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
       
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5,eps=1e-6)
        total_steps= EPOCHS*len(train_loader)
        warmup_steps= 0.1*total_steps
        scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps,
            )

        best_auc=0
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            loss = train_one_epoch(model, train_loader, optimizer, scheduler)
            val_auc, val_loss ,val_preds = validate(model, val_loader)
                        
            print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_auc.bin")
    
        all_preds.append(pd.Series(val_preds))
    
    # Clean up paraphrasing model
    if USE_PARAPHRASING and paraphrase_model is not None:
        del paraphrase_model, paraphrase_tokenizer
        torch.cuda.empty_cache()
        print("Paraphrasing model cleaned up")

what to do at test time, dont take examples from test set into val.

In [17]:
# all_truths=[]
# all_rules=[]
# all_preds=[]

# all_truths.append(val.label.apply(lambda x:x>=.5).astype('float'))
# all_rules.append(val.rule)
# wts=torch.load(f"model_fold{0}_auc.bin", map_location=DEVICE)
# wts= {k.replace('module.',''):v for k,v in wts.items()}
        
# model.load_state_dict(wts)
# model.eval()
# val_ds = JigsawDataset(
#         val['text'].tolist(), 
#         val['label'].tolist(), 
#         tokenizer, 
#         MAX_LEN*2
#     )
    
# val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# _,_,val_preds= validate(model,val_loader)
        
# all_preds.append(pd.Series(val_preds))
            

# preddf= pd.DataFrame(columns=['preds','truths','rule'])
# preddf.preds=pd.concat(all_preds,ignore_index=True)
# preddf.rule= pd.concat(all_rules,ignore_index=True)
# preddf.truths= pd.concat(all_truths,ignore_index=True)

# print(preddf.groupby('rule').apply(lambda group: roc_auc_score(group['truths'],group['preds'])))

In [18]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts=torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
            
        model.load_state_dict(wts)
        model.eval()
    
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test),[0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)

In [19]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv